[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/04_grounded_data_augmentation.ipynb)

# Step 4 — Instruction Back-Translation

Generate training Q&A from TRAIN paragraphs using **instruction back-translation**:
given passage text *y*, ask a teacher model for a question/instruction *x* for which
*y* is a good answer ([Li et al.](https://openreview.net/forum?id=1oijHJBRsT)).

## Learning objectives
- Apply instruction back-translation to policy passages
- Build a faithful SFT corpus without free-form hallucination
- Inspect heuristic rejections and compare to the notebook-03 filter path


Step 2 data teaches how to prompt; Step 4 data is what you train on
Back-translation + overlap check reduces hallucinated training labels
In a real deployment you'd have more documents and actually hit 500–1000; the bootcamp simulates the pipeline at small scale


In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    DEFAULT_SYNTHETIC_TARGET_SIZE,
    PARAGRAPHS_PATH,
    SYNTHETIC_IBT_PATH,
    Paragraph,
    ParagraphSplit,
    QASample,
    apply_heuristic_filters,
    create_teacher_client,
    effective_synthetic_target,
    generate_grounded_training_corpus,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    summarize_heuristic_rejections,
    use_repo_root,
)
from rich.console import Console
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


## 1. Load TRAIN paragraphs (exclude test holdout)

In [2]:
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
train_paragraphs = [p for p in all_paragraphs if p.split == ParagraphSplit.TRAIN]
console.print(f"Train paragraphs: {len(train_paragraphs)}")

Train paragraphs: 36

## 2. Generate Q&A via instruction back-translation


So notebook 02 → 03 walks you through the canonical "generate everything, then filter with heuristics + judge" pipeline as a learning exercise. Notebook 04 is the alternative: generate questions for which each train paragraph *is* the answer (instruction back-translation), so grounding is built into the data-creation step. The training file you already saved came from the 02→03 path; notebook 04 produces a different corpus with this method.


`rag.py` implements instruction back-translation (prompts live in `prompts.py` so they are easy to edit):

1. The teacher is asked **only for a question** answered by the passage (`instruction_backtranslation_prompt`). The gold answer defaults to the passage text itself.
2. A post-hoc lexical overlap check (`grounding_overlap_score`) still rejects samples whose answer tokens do not sufficiently appear in the passage (useful if a short extractive answer is returned).

We request **at most one sample per paragraph**. Cycling the same paragraph with the same prompt mostly regenerates duplicates that heuristics then discard.


You treat the passage as the answer and only ask the teacher for a question that that text would satisfy. So:

- **Less hallucination in the label** — the training target is (mostly) real document text, not a model-written “gold” answer that might be wrong.
- **Clearer method** — it’s a known technique (question from answer), distinct from ([nb02](implementations/qa_text_generation/02_synthetic_qa_generation.ipynb)) free-form generation strategies. Nb02 explores prompting; nb04 builds a trainable corpus with a grounding guarantee.
- **Cheaper / less waste** — one question per paragraph, no cycling identical prompts, so you don’t burn teacher calls on duplicates heuristics would drop anyway.
- It also **reduces shallow-answer risk** you get when an LLM generates both sides of the pair.
- **Natural writing style/length** - which can help a model sound less "templated" than one trained purely on synthetic Q&A.
### Tradeoff (worth knowing)
Answers are often long and extractive (the passage itself), so the SLM is trained more toward “quote/recall the policy” than “short crisp answers.” That’s intentional for grounding; if you later want short answers, you’d add a second step (e.g. compress the passage into a brief gold answer) — still grounded, but not pure classic back-translation.

In [3]:
teacher = create_teacher_client()
target_size = effective_synthetic_target(
    train_paragraphs,
    requested=DEFAULT_SYNTHETIC_TARGET_SIZE,
    one_per_paragraph=True,
)
print(f"Back-translation generation target: {target_size} (cap = # train paragraphs)")

grounded_candidates = generate_grounded_training_corpus(
    teacher,
    train_paragraphs,
    target_size=target_size,
    min_overlap=0.15,
)
print(f"Generated {len(grounded_candidates)} back-translated candidates")
grounded_candidates[0].metadata if grounded_candidates else {}

Back-translation generation target: 36 (cap = # train paragraphs)
Generated 36 back-translated candidates


{'generation_strategy': 'instruction_backtranslation',
 'grounding_overlap': 1.0}

## 3. Filter and save final SFT corpus


In [4]:
final_samples, rejected = apply_heuristic_filters(grounded_candidates)
print(f"Final SFT corpus size: {len(final_samples)} (rejected {len(rejected)})")
print("Rejection reasons:", summarize_heuristic_rejections(rejected))

table = Table(title="Sample heuristic rejections (notebook 04)", show_lines=True)
table.add_column("id", style="cyan", max_width=20)
table.add_column("reason(s)", style="red")
table.add_column("question", overflow="fold")
for row in rejected[:8]:
    table.add_row(row.get("id", ""), row.get("reasons", row.get("reason", "")), row.get("question", ""))
if rejected:
    console.print(table)
else:
    print("No heuristic rejections — expected when generating one sample per paragraph.")

save_typed_jsonl(
    SYNTHETIC_IBT_PATH,
    final_samples,
    to_dict=QASample.to_dict,
)
SYNTHETIC_IBT_PATH

Final SFT corpus size: 36 (rejected 0)
Rejection reasons: {}
No heuristic rejections — expected when generating one sample per paragraph.


PosixPath('/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/synthetic_back_translation.jsonl')